# Classificação Ordinal com Sentence Embeddings (Sentence-BERT)

## Motivação

Neste experimento combinamos duas ideias:

1. **Sentence Embeddings (Sentence-BERT):** em vez de representar o texto por contagem de palavras (TF-IDF) ou características manuais, usamos uma rede neural pré-treinada para transformar cada resposta em um vetor numérico denso que captura o **significado semântico** do texto.
2. **Classificação Ordinal ($c1 < c234 < c5$):** os rótulos de clareza possuem uma ordem natural:
   $$c1 \text{ (menos clara)} < c234 \text{ (intermediária)} < c5 \text{ (mais clara)}$$
   Um classificador multiclasse comum trata as três classes como categorias independentes (errar $c1$ chutando $c5$ tem o mesmo peso que errar chutando $c234$). A classificação ordinal incorpora explicitamente essa relação de ordem.

## Passo 1: Importando bibliotecas e carregando os dados pré-processados

Utilizamos os conjuntos com a limpeza básica do projeto (`train_limpo.csv`, `test1_limpo.csv` e `test2_limpo.csv`).

- `train_limpo.csv`: usado para treinar e escolher hiperparâmetros (via Grid Search + validação cruzada de 5 folds).
- `test1_limpo.csv`: usado **apenas** ao final para avaliação e comparação estatística (McNemar) com o modelo do EP1.
- `test2_limpo.csv`: conjunto oficial do EP2 (sem rótulos).

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

PASTA_DADOS = Path("data") if Path("data").exists() else Path("../data")

train = pd.read_csv(PASTA_DADOS / "train_limpo.csv")
test1 = pd.read_csv(PASTA_DADOS / "test1_limpo.csv")
test2 = pd.read_csv(PASTA_DADOS / "test2_limpo.csv")

print("train:", train.shape)
print("test1:", test1.shape)
print("test2:", test2.shape)

train: (20092, 2)
test1: (900, 2)
test2: (897, 2)


### Conferindo os dados e codificando a ordem dos rótulos ($c1 < c234 < c5$)

Para que o classificador ordinal entenda a hierarquia entre as classes, mapeamos os rótulos para números inteiros ordenados:
- `c1` $\rightarrow 0$
- `c234` $\rightarrow 1$
- `c5` $\rightarrow 2$

In [3]:
# Garantindo que não há valores nulos no texto
for nome, df in [("train", train), ("test1", test1), ("test2", test2)]:
    df["resp_text"] = df["resp_text"].fillna("").astype(str)
    print(f"{nome}: {len(df)} linhas | textos vazios: {(df['resp_text'] == '').sum()}")

# Mapeamento ordinal: c1 < c234 < c5
ORDEM_CLASSES = {"c1": 0, "c234": 1, "c5": 2}
INV_CLASSES = {0: "c1", 1: "c234", 2: "c5"}

y_train = train["clarity"].map(ORDEM_CLASSES).values
y_test1 = test1["clarity"].map(ORDEM_CLASSES).values

print("\nDistribuição no treino (0=c1, 1=c234, 2=c5):")
print(pd.Series(y_train).value_counts().sort_index())
train.head()

train: 20092 linhas | textos vazios: 0
test1: 900 linhas | textos vazios: 0
test2: 897 linhas | textos vazios: 0

Distribuição no treino (0=c1, 1=c234, 2=c5):
0    6347
1    6853
2    6892
Name: count, dtype: int64


,resp_text,clarity
0,prezado a senhor a esclarecemos que o serviço ...,c5
1,prezada cidadã as informações sobre óbitos no ...,c1
2,prezado senhor julio a ouvidoria geral da petr...,c1
3,prezado a senhor a esclarecemos que o serviço ...,c234
4,senhor o serviço de informações ao cidadão do ...,c234


## Passo 2: Geração dos Sentence Embeddings (Sentence-BERT)

Utilizamos o modelo pré-treinado **`paraphrase-multilingual-MiniLM-L12-v2`** da biblioteca `sentence-transformers`:
- Foi treinado em mais de 50 idiomas (incluindo português) para representar textos com significados semelhantes em pontos próximos de um espaço de **384 dimensões**.
- Como o modelo é pré-treinado, a extração dos embeddings **não usa os rótulos** (zero risco de vazamento de dados).
- Os vetores são normalizados (`normalize_embeddings=True`) e salvos no arquivo `embeddings_sbert.npz`. Assim, se você reiniciar o notebook depois, ele carrega os vetores prontos em menos de 1 segundo em vez de recalcular tudo.

In [4]:
import time
from sentence_transformers import SentenceTransformer

ARQUIVO_EMB = PASTA_DADOS.parent / "embeddings_sbert.npz"
NOME_MODELO_SBERT = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

if ARQUIVO_EMB.exists():
    print(f"Carregando embeddings já salvos em {ARQUIVO_EMB}...")
    dados_emb = np.load(ARQUIVO_EMB)
    X_train_emb = dados_emb["X_train"]
    X_test1_emb = dados_emb["X_test1"]
    X_test2_emb = dados_emb["X_test2"]
else:
    print(f"Baixando/carregando modelo {NOME_MODELO_SBERT}...")
    sbert = SentenceTransformer(NOME_MODELO_SBERT)
    inicio = time.time()

    print("Gerando embeddings do treino (20.092 textos)...")
    X_train_emb = sbert.encode(
        train["resp_text"].tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    print("Gerando embeddings do teste1 (900 textos)...")
    X_test1_emb = sbert.encode(
        test1["resp_text"].tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    print("Gerando embeddings do teste2 (897 textos)...")
    X_test2_emb = sbert.encode(
        test2["resp_text"].tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
    )

    np.savez_compressed(
        ARQUIVO_EMB,
        X_train=X_train_emb,
        X_test1=X_test1_emb,
        X_test2=X_test2_emb,
    )
    print(f"Concluído em {(time.time() - inicio) / 60:.1f} min! Salvo em {ARQUIVO_EMB}.")

print("Shapes dos embeddings:", X_train_emb.shape, X_test1_emb.shape, X_test2_emb.shape)

Carregando embeddings já salvos em ../embeddings_sbert.npz...
Shapes dos embeddings: (20092, 384) (900, 384) (897, 384)


## Passo 3: Classificadores Ordinais e Grid Search (5 folds)

Para aproveitar a estrutura ordenada dos rótulos ($c1 < c234 < c5$, codificados como $0 < 1 < 2$), avaliamos **duas formulações clássicas de classificação ordinal**:

1. **Decomposição Binária Cumulativa de Frank & Hall (2001) (`OrdinalFrankHallClassifier`):**
   - Transforma o problema de 3 classes ordenadas em **2 classificadores binários** que respeitam a ordem:
     - Classificador 1: a resposta tem clareza **maior que $c1$**? ($y > 0$, ou seja, $\{c234, c5\}$ vs. $\{c1\}$)
     - Classificador 2: a resposta tem clareza **maior que $c234$**? ($y > 1$, ou seja, $\{c5\}$ vs. $\{c1, c234\}$)
   - As probabilidades das 3 classes são reconstruídas a partir das probabilidades cumulativas:
     $$P(c1) = 1 - P(y > c1), \quad P(c234) = P(y > c1) - P(y > c234), \quad P(c5) = P(y > c234)$$

2. **Regressão Logística Ordinal de Limiares Cumulativos — *Proportional Odds / Cumulative Link* (`OrdinalThresholdClassifier`):**
   - Aprende um **único vetor de pesos** $w \in \mathbb{R}^{384}$ (que define uma única direção de "clareza" no espaço semântico dos embeddings) e dois limiares ordenados $\theta_1 < \theta_2$:
     $$P(y \le k \mid x) = \sigma(\theta_k - w^\top x)$$
   - O produto escalar $s(x) = w^\top x$ funciona como um **índice contínuo de clareza** para cada resposta: se $s(x) < \theta_1$, prevê $c1$; se $\theta_1 \le s(x) < \theta_2$, prevê $c234$; se $s(x) \ge \theta_2$, prevê $c5$.

Em ambos os modelos, testamos no Grid Search duas regras de decisão final (`modo_decisao`):
- `"prob"`: escolhe a classe de maior probabilidade $\arg\max_k P(y=k \mid x)$;
- `"esperanca"`: calcula a esperança matemática ordinal $E[y \mid x] = 0\cdot P(c1) + 1\cdot P(c234) + 2\cdot P(c5)$ e arredonda para a classe mais próxima.

In [5]:
import numpy as np
from scipy.optimize import minimize
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold


class OrdinalFrankHallClassifier(BaseEstimator, ClassifierMixin):
    """Classificador Ordinal Cumulativo de Frank & Hall (2001).

    Para 3 classes ordenadas (0 < 1 < 2, isto é, c1 < c234 < c5), treina 2
    classificadores binários com regularização L2:
      - Classificador 1: P(y > 0) = P(y in {1, 2})
      - Classificador 2: P(y > 1) = P(y == 2)
    E reconstrói as probabilidades respeitando a ordem:
      - P(y == 0) = 1 - P(y > 0)
      - P(y == 1) = max(0, P(y > 0) - P(y > 1))
      - P(y == 2) = P(y > 1)
    """

    def __init__(self, C=1.0, class_weight=None, modo_decisao="prob"):
        self.C = C
        self.class_weight = class_weight
        self.modo_decisao = modo_decisao  # "prob" (argmax) ou "esperanca" (valor esperado ordinal)

    def fit(self, X, y):
        y = np.asarray(y, dtype=int)
        self.classes_ = np.unique(y)
        self.modelos_ = []
        for limiar in self.classes_[:-1]:
            y_bin = (y > limiar).astype(int)
            clf = LogisticRegression(
                C=self.C,
                class_weight=self.class_weight,
                max_iter=1000,
                random_state=123,
            )
            clf.fit(X, y_bin)
            self.modelos_.append(clf)
        return self

    def predict_proba(self, X):
        # p_maior[:, k] = P(y > k)
        p_maior = np.column_stack([m.predict_proba(X)[:, 1] for m in self.modelos_])
        n = X.shape[0]
        probs = np.zeros((n, len(self.classes_)))
        probs[:, 0] = 1.0 - p_maior[:, 0]
        for k in range(1, len(self.classes_) - 1):
            probs[:, k] = np.maximum(0.0, p_maior[:, k - 1] - p_maior[:, k])
        probs[:, -1] = p_maior[:, -1]
        soma = probs.sum(axis=1, keepdims=True)
        return probs / np.maximum(soma, 1e-12)

    def predict(self, X):
        probs = self.predict_proba(X)
        if self.modo_decisao == "esperanca":
            # Valor esperado E[y] = 0*P(0) + 1*P(1) + 2*P(2), arredondado para {0, 1, 2}
            esperado = probs @ self.classes_
            return np.clip(np.rint(esperado), self.classes_[0], self.classes_[-1]).astype(int)
        return self.classes_[np.argmax(probs, axis=1)]


class OrdinalThresholdClassifier(BaseEstimator, ClassifierMixin):
    """Regressão Logística Ordinal de Limiares Cumulativos (Proportional Odds / Cumulative Link).

    Aprende um único vetor de pesos w (direção semântica de clareza) e dois limiares
    ordenados theta_1 < theta_2 tais que:
        P(y <= k | x) = sigmoid(theta_k - w^T x)
    """

    def __init__(self, alpha=1.0, class_weight=None, modo_decisao="prob"):
        self.alpha = alpha
        self.class_weight = class_weight
        self.modo_decisao = modo_decisao

    def fit(self, X, y):
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y, dtype=int)
        self.classes_ = np.unique(y)
        n, d = X.shape

        if self.class_weight == "balanced":
            cont = np.bincount(y)
            pesos_classe = n / (len(self.classes_) * cont)
            amostra_peso = pesos_classe[y]
        else:
            amostra_peso = np.ones(n, dtype=np.float64)

        # Parametrização: params = [w (d), theta_1, log_delta] onde theta_2 = theta_1 + exp(log_delta)
        params0 = np.zeros(d + 2, dtype=np.float64)
        params0[d] = -0.5
        params0[d + 1] = 0.0

        y0 = (y == 0)
        y1 = (y == 1)
        y2 = (y == 2)

        def loss_and_grad(params):
            w = params[:d]
            t1 = params[d]
            delta = np.exp(np.clip(params[d + 1], -20, 20))
            t2 = t1 + delta

            z = X @ w
            s1 = (1.0 / (1.0 + np.exp(-np.clip(t1 - z, -40, 40))))
            s2 = (1.0 / (1.0 + np.exp(-np.clip(t2 - z, -40, 40))))

            p0 = np.clip(s1, 1e-12, 1.0)
            p1 = np.clip(s2 - s1, 1e-12, 1.0)
            p2 = np.clip(1.0 - s2, 1e-12, 1.0)

            nll = -np.sum(amostra_peso * (y0 * np.log(p0) + y1 * np.log(p1) + y2 * np.log(p2))) / n
            reg = 0.5 * self.alpha * np.dot(w, w) / n

            # Derivadas em relação a a1 = t1 - z e a2 = t2 - z
            ds1 = s1 * (1.0 - s1)
            ds2 = s2 * (1.0 - s2)

            g_a1 = np.zeros(n)
            g_a2 = np.zeros(n)

            g_a1[y0] = -(1.0 - s1[y0])
            g_a1[y1] = ds1[y1] / p1[y1]
            g_a2[y1] = -ds2[y1] / p1[y1]
            g_a2[y2] = s2[y2]

            g_a1 *= amostra_peso / n
            g_a2 *= amostra_peso / n

            grad_w = -X.T @ (g_a1 + g_a2) + (self.alpha / n) * w
            grad_t1 = np.sum(g_a1 + g_a2)
            grad_log_delta = np.sum(g_a2) * delta
            return nll + reg, np.concatenate([grad_w, [grad_t1, grad_log_delta]])

        res = minimize(loss_and_grad, params0, jac=True, method="L-BFGS-B", options={"maxiter": 500})
        self.coef_ = res.x[:d]
        t1 = res.x[d]
        t2 = t1 + np.exp(np.clip(res.x[d + 1], -20, 20))
        self.thresholds_ = np.array([t1, t2])
        return self

    def decision_function(self, X):
        """Score escalar contínuo de clareza (w^T x)."""
        return np.asarray(X, dtype=np.float64) @ self.coef_

    def predict_proba(self, X):
        z = self.decision_function(X)
        s1 = (1.0 / (1.0 + np.exp(-np.clip(self.thresholds_[0] - z, -40, 40))))
        s2 = (1.0 / (1.0 + np.exp(-np.clip(self.thresholds_[1] - z, -40, 40))))
        p0 = s1
        p1 = np.maximum(0.0, s2 - s1)
        p2 = 1.0 - s2
        probs = np.column_stack([p0, p1, p2])
        return probs / np.maximum(probs.sum(axis=1, keepdims=True), 1e-12)

    def predict(self, X):
        probs = self.predict_proba(X)
        if self.modo_decisao == "esperanca":
            esperado = probs @ self.classes_
            return np.clip(np.rint(esperado), self.classes_[0], self.classes_[-1]).astype(int)
        return self.classes_[np.argmax(probs, axis=1)]


### Executando o Grid Search com Validação Cruzada (5 folds)

Seguindo as regras do trabalho:
- Usamos `StratifiedKFold` com **5 folds** e `random_state=123` (apenas no treino).
- Testamos hiperparâmetros para os dois classificadores ordinais e também avaliamos um **classificador multiclasse tradicional** (`LogisticRegression` multinomial/softmax) sobre os mesmos embeddings, para verificar diretamente se a modelagem ordinal melhora o resultado.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)

# 1) Grid Search: Classificador Ordinal de Frank & Hall
param_grid_fh = {
    "C": [0.05, 0.1, 0.25, 0.5, 1.0, 2.0, 5.0, 10.0],
    "class_weight": [None, "balanced"],
    "modo_decisao": ["prob", "esperanca"],
}
grid_fh = GridSearchCV(
    OrdinalFrankHallClassifier(),
    param_grid_fh,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
    return_train_score=True,
)
grid_fh.fit(X_train_emb, y_train)
print("=== Ordinal (Frank & Hall) ===")
print("Melhores parâmetros:", grid_fh.best_params_)
print(f"Acurácia CV (5 folds): {grid_fh.best_score_:.4f}\n")

# 2) Grid Search: Regressão Ordinal de Limiares Cumulativos (Proportional Odds)
param_grid_th = {
    "alpha": [0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 25.0],
    "class_weight": [None, "balanced"],
    "modo_decisao": ["prob", "esperanca"],
}
grid_th = GridSearchCV(
    OrdinalThresholdClassifier(),
    param_grid_th,
    cv=cv,
    scoring="accuracy",
    n_jobs=1,
    return_train_score=True,
)
grid_th.fit(X_train_emb, y_train)
print("=== Ordinal (Limiares Cumulativos / Proportional Odds) ===")
print("Melhores parâmetros:", grid_th.best_params_)
print(f"Acurácia CV (5 folds): {grid_th.best_score_:.4f}\n")

# 3) Comparação de referência: Classificador Multiclasse Tradicional nos mesmos embeddings
param_grid_mc = {
    "C": [0.05, 0.1, 0.25, 0.5, 1.0, 2.0, 5.0, 10.0],
    "class_weight": [None, "balanced"],
}
grid_mc = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=123),
    param_grid_mc,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
)
grid_mc.fit(X_train_emb, y_train)
print("=== Multiclasse Tradicional (mesmos embeddings) ===")
print("Melhores parâmetros:", grid_mc.best_params_)
print(f"Acurácia CV (5 folds): {grid_mc.best_score_:.4f}")

### Comparativo na validação cruzada

In [ ]:
resumo_cv = pd.DataFrame([
    {
        "Modelo": "Ordinal - Frank & Hall (Sentence-BERT)",
        "Acurácia CV (5 folds)": grid_fh.best_score_,
        "Melhores hiperparâmetros": str(grid_fh.best_params_),
    },
    {
        "Modelo": "Ordinal - Limiares Cumulativos (Sentence-BERT)",
        "Acurácia CV (5 folds)": grid_th.best_score_,
        "Melhores hiperparâmetros": str(grid_th.best_params_),
    },
    {
        "Modelo": "Multiclasse Tradicional (Sentence-BERT)",
        "Acurácia CV (5 folds)": grid_mc.best_score_,
        "Melhores hiperparâmetros": str(grid_mc.best_params_),
    },
    {
        "Modelo": "EP1 - Regressão Logística + TF-IDF (referência)",
        "Acurácia CV (5 folds)": 0.4632,
        "Melhores hiperparâmetros": "baseline EP1",
    },
]).sort_values("Acurácia CV (5 folds)", ascending=False)

resumo_cv

## Resultados obtidos e conclusão

| Modelo | Acurácia CV (5 folds) | Melhores hiperparâmetros |
|---|---|---|
| EP1 - Regressão Logística + TF-IDF (referência) | **0,4632** | baseline EP1 |
| Multiclasse Tradicional (Sentence-BERT) | 0,4332 | `{'C': 1.0, 'class_weight': 'balanced'}` |
| Ordinal - Frank & Hall (Sentence-BERT) | 0,4300 | `{'C': 0.5, 'class_weight': None, 'modo_decisao': 'prob'}` |
| Ordinal - Limiares Cumulativos (Sentence-BERT) | 0,4208 | `{'alpha': 0.5, 'class_weight': None, 'modo_decisao': 'prob'}` |

**Conclusão:** Os classificadores ordinais com Sentence Embeddings (`0,4300` e `0,4208`) não superaram nem o classificador multiclasse tradicional sobre os mesmos embeddings (`0,4332`) nem o baseline do EP1 (`0,4632`). Por ficar abaixo do baseline na validação cruzada, o modelo não foi selecionado como modelo final.